# Causal Inference in Practice
## Week 1 — The Causal Question · Practice Notebook

> **Block I — Foundations**
>
> Why prediction is not intervention, why correlation is not causation, and the vocabulary that makes the difference precise.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · Prediction is not intervention

We'll build the umbrella/rain intuition in code. Umbrellas are *seen* before rain, so they **predict** rain — but handing out umbrellas does nothing to the weather. The point of this notebook is to feel the difference between **conditioning** (`P(Y|X)`) and **intervening** (`P(Y|do(X))`) on simulated data where we *know* the truth.

In [ ]:
# A tiny world: clouds cause both umbrellas and rain.
n = 100_000
clouds = RNG.binomial(1, 0.3, n)            # 30% of days are cloudy
umbrella = RNG.binomial(1, 0.05 + 0.8*clouds, n)  # people react to clouds
rain = RNG.binomial(1, 0.05 + 0.7*clouds, n)      # clouds cause rain
df = pd.DataFrame({'clouds': clouds, 'umbrella': umbrella, 'rain': rain})

# Seeing: P(rain | umbrella) — umbrellas 'predict' rain
p_seen = df.groupby('umbrella')['rain'].mean()
print('P(rain | umbrella seen):')
print(p_seen)
print(f"\nNaive 'effect' of umbrellas on rain: "
      f"{p_seen[1] - p_seen[0]:+.3f}")

Umbrellas look strongly associated with rain. Now **intervene**: force everyone to carry an umbrella (`do(umbrella=1)`) vs nobody (`do(umbrella=0)`). Because nothing downstream of clouds touches the weather, the rain rate is unchanged — the *causal* effect is ~0.

In [ ]:
# do(umbrella): break the arrow clouds -> umbrella by setting it.
# Rain only depends on clouds, so intervening on umbrella changes nothing.
rain_do1 = (0.05 + 0.7*clouds).mean()   # everyone carries an umbrella
rain_do0 = (0.05 + 0.7*clouds).mean()   # no one does
print(f'P(rain | do(umbrella=1)) = {rain_do1:.3f}')
print(f'P(rain | do(umbrella=0)) = {rain_do0:.3f}')
print(f'Causal effect of umbrellas on rain: {rain_do1 - rain_do0:+.3f}')
print('\nSeeing said ~+0.5; doing says 0. That gap is confounding.')

## 2 · Hello, confounding (the eight-line course)

A confounder `C` pushes both treatment `X` and outcome `Y`. The **true** effect of `X` on `Y` is exactly `2`. A naive regression credits some of `C`'s effect to `X`; adjusting for `C` recovers the truth.

In [ ]:
import statsmodels.api as sm

n = 5000
C = RNG.normal(size=n)                       # confounder
X = 0.8 * C + RNG.normal(size=n)             # C pushes treatment
Y = 2 * X + 1.5 * C + RNG.normal(size=n)     # TRUE effect of X is 2

naive    = sm.OLS(Y, sm.add_constant(X)).fit().params[1]
adjusted = sm.OLS(Y, sm.add_constant(np.c_[X, C])).fit().params[1]
print(f'naive    = {naive:.3f}   (biased upward)')
print(f'adjusted = {adjusted:.3f}   (~2.0, the truth)')

### 🔧 Exercise 2.1 — make the bias worse, then fix it

Change the confounder's strength so it pushes `X` **and** `Y` harder (e.g. `X = 1.5*C + noise`, `Y = 2*X + 3*C + noise`). Predict whether the naive estimate goes up or down *before* you run it. Then confirm the adjusted estimate still recovers ~2.

Fill in the `# TODO`s below.

In [ ]:
# TODO: set stronger confounding and re-estimate.
C2 = RNG.normal(size=n)
X2 = ...        # TODO: make C push X harder
Y2 = ...        # TODO: true effect of X2 still 2, but C pushes Y harder
# naive2    = ...
# adjusted2 = ...
# print(naive2, adjusted2)

### ✅ Solution 2.1

In [ ]:
C2 = RNG.normal(size=n)
X2 = 1.5 * C2 + RNG.normal(size=n)
Y2 = 2 * X2 + 3.0 * C2 + RNG.normal(size=n)
naive2    = sm.OLS(Y2, sm.add_constant(X2)).fit().params[1]
adjusted2 = sm.OLS(Y2, sm.add_constant(np.c_[X2, C2])).fit().params[1]
print(f'naive2    = {naive2:.3f}   (more bias than before)')
print(f'adjusted2 = {adjusted2:.3f}   (still ~2.0)')
assert abs(adjusted2 - 2) < 0.15, 'adjustment should recover ~2'

## 3 · Confounder vs. mediator vs. collider

Same three-node shapes, three different correct actions. We simulate each and watch what 'controlling for' the third variable does to the estimated `X → Y` effect. **In all three, the true direct effect of `X` on `Y` is 1.**

In [ ]:
def est(y, *cols):
    """OLS slope on the first regressor (X), adjusting for the rest."""
    Xmat = sm.add_constant(np.column_stack(cols))
    return sm.OLS(y, Xmat).fit().params[1]

n = 8000

# --- CONFOUNDER:  Z -> X, Z -> Y  (adjust!) ---
Z = RNG.normal(size=n)
Xc = 1.0*Z + RNG.normal(size=n)
Yc = 1.0*Xc + 2.0*Z + RNG.normal(size=n)     # true X->Y = 1
print('CONFOUNDER  naive=%.2f  adjusted=%.2f  (truth 1.0)' %
      (est(Yc, Xc), est(Yc, Xc, Z)))

In [ ]:
# --- MEDIATOR:  X -> M -> Y  (do NOT adjust for total effect) ---
Xm = RNG.normal(size=n)
M  = 1.0*Xm + RNG.normal(size=n)
Ym = 1.0*M + RNG.normal(size=n)              # total X->Y = 1 (via M)
print('MEDIATOR    total=%.2f  over-adjusted=%.2f  (truth 1.0)' %
      (est(Ym, Xm), est(Ym, Xm, M)))
print('  -> adjusting for M wrongly removes the effect.\n')

# --- COLLIDER:  X -> K <- Y  (do NOT adjust) ---
Xk = RNG.normal(size=n)
Yk = 1.0*Xk + RNG.normal(size=n)             # true X->Y = 1
K  = 1.0*Xk + 1.0*Yk + RNG.normal(size=n)    # common effect
print('COLLIDER    clean=%.2f  adjusted=%.2f  (truth 1.0)' %
      (est(Yk, Xk), est(Yk, Xk, K)))
print('  -> adjusting for K opens a spurious path and biases the estimate.')

Notice the pattern: **adjusting helped only the confounder.** For the mediator and collider it *hurt*. This is why 'control for everything' is wrong — and why we draw the graph first.

### 🔧 Exercise 3.1 — M-bias

Build the M-bias structure from Problem Set 1 #4: hidden `U1 → X`, `U2 → Y`, and a pre-treatment `Z` with `U1 → Z ← U2`. There is **no** real confounding, so the naive `X → Y` estimate is already unbiased. Show that adjusting for the innocent-looking pre-treatment `Z` *introduces* bias.

In [ ]:
# TODO: simulate U1, U2, then X, Y, and collider Z = U1 + U2 + noise.
# true effect of X on Y here is 1.0
# U1 = ...
# U2 = ...
# Xz = 1.0*U1 + RNG.normal(size=n)
# Yz = 1.0*Xz + 1.0*U2 + RNG.normal(size=n)
# Z  = ...   # collider of U1 and U2
# print(est(Yz, Xz), est(Yz, Xz, Z))

### ✅ Solution 3.1

In [ ]:
U1 = RNG.normal(size=n)
U2 = RNG.normal(size=n)
Xz = 1.0*U1 + RNG.normal(size=n)
Yz = 1.0*Xz + 1.0*U2 + RNG.normal(size=n)     # true X->Y = 1
Z  = 1.0*U1 + 1.0*U2 + RNG.normal(size=n)      # M-bias collider
print('M-BIAS  unadjusted=%.2f  adjusted-for-Z=%.2f  (truth 1.0)' %
      (est(Yz, Xz), est(Yz, Xz, Z)))
print('Pre-treatment did NOT mean safe: adjusting for Z added bias.')

## 4 · Simpson's paradox in code (kidney stones)

We reconstruct the famous table and watch an effect flip when we aggregate across stone size — the confounder.

In [ ]:
tab = pd.DataFrame({
    'size':      ['small','small','large','large'],
    'treatment': ['A','B','A','B'],
    'success':   [81, 234, 192, 55],
    'n':         [87, 270, 263, 80]})
tab['rate'] = tab['success'] / tab['n']

print('Within each stone size (A beats B both times):')
print(tab.pivot(index='size', columns='treatment', values='rate'), '\n')

combined = tab.groupby('treatment').apply(
    lambda g: g['success'].sum() / g['n'].sum(), include_groups=False)
print('Combined (B beats A!):')
print(combined)

Doctors gave Treatment A (open surgery) to the harder large-stone cases. **Stone size is a confounder of treatment and success**, so the subgroup numbers — not the combined ones — answer the causal question. The table alone can't tell you that; the *causal story* does.

### 🔧 Exercise 4.1 — which number would you report?

Suppose instead the third variable were a **mediator** (treatment → blood pressure → cure) rather than a confounder. Which number — subgroup or combined — should you trust then? Write your answer in the next cell as a comment, then reveal the solution.

In [ ]:
# Your answer here:
# ...

### ✅ Solution 4.1

If the splitting variable is a **mediator** on the causal path, you want the **combined** (unadjusted) estimate — conditioning on a mediator removes part of the very effect you're after. Same table, opposite advice: only the DAG decides. (This is exactly the contrast on the 'The DAG decides' lecture slide.)

## 5 · Wrap-up & self-check

- `P(Y|X)` ≠ `P(Y|do(X))` whenever a back-door path is open.
- **Confounder → adjust; mediator → don't (for total effect); collider → don't.**
- Adjusting for the wrong variable *creates* bias — you saw it in code for both the collider and M-bias.
- Simpson's paradox is confounding in disguise; the causal model picks the right number.

**You're ready for Week 2** if you can reproduce the naive-vs-adjusted gap from memory and call confounder / mediator / collider on sight. Next week: potential outcomes `Y(1), Y(0)` and the assumptions that license a causal claim.